# 北京案例 2021–2025 嵌入变化比较 修订版

请在 **Python (tourism_embed)** 内核中从上到下运行。

本版保留原始北京AOI、年份、64维、10米网格及评分公式。只调整依赖检查、网络读取配置、GIS接口和进度提示，并增加可选结果导出。

已针对此前的Zarr 2版本和远程超时问题修改；**未完成远程档案全流程重跑，不能保证当地网络下载成功**。输出用于探索性变化筛查，不是土地用途、因果或合规性验证。

原始平台导出文件应另存保留；本文件属于修改后的运行版本。请勿用它替换GitHub v0.1.0中的原始导出文件并继续称其为“未修改原件”。


## 1 依赖安装（已有tourism_embed环境可跳过）

In [ ]:
# 如首次运行提示缺包，可取消下一行开头的 # 后运行。
# 安装结束后重启内核，再从环境检查单元格继续。
# %pip install "numpy>=2" "zarr>=3.3,<4" obstore matplotlib pyproj rasterio "shapely>=2.1,<3" Pillow ipykernel


## 2 环境检查

In [ ]:
import sys
import importlib
from importlib.metadata import version

print("Python:", sys.executable)
for module in ("numpy", "zarr", "obstore", "matplotlib", "pyproj", "rasterio", "shapely"):
    try:
        importlib.import_module(module)
        print(f"{module}: {version(module)}")
    except ImportError as exc:
        raise RuntimeError(
            f"当前内核缺少或无法导入 {module}。请在上面的安装单元格安装依赖，重启内核后重试。"
        ) from exc

import zarr
if not hasattr(zarr.storage, "ObjectStore"):
    raise RuntimeError("当前Zarr缺少ObjectStore。请确认内核为tourism_embed并安装zarr>=3.3,<4，再重启内核。")
if tuple(int(x) for x in version("zarr").split(".")[:2]) < (3, 3):
    raise RuntimeError("本修订版需要Zarr>=3.3,<4，以支持下方的读取配置。")
if tuple(int(x) for x in version("shapely").split(".")[:2]) < (2, 1):
    raise RuntimeError("本修订版使用Shapely>=2.1，请更新后重启内核。")


## 3 输入参数和网络设置

In [ ]:
# AOI coordinates are [longitude, latitude].
AOI = [[[116.65913992370832,39.84760743125574],[116.68936378050216,39.84760743125574],[116.68936378050216,39.86185706107452],[116.65913992370832,39.86185706107452],[116.65913992370832,39.84760743125574]]]
YEARS = [2021,2025]
VIEW = "pair"
EMBEDDING_DIMS = 64 # Use 128 for the full embedding
STORE_URL = "s3://olmoearth/olmoearth-v1-3-embeddings/embeddings.zarr"
STORE_ENDPOINT = "https://data.source.coop"

# 网络配置：这些参数不改变评分公式。
ASYNC_CONCURRENCY = 1
CONNECT_TIMEOUT_SECONDS = 20
REQUEST_TIMEOUT_SECONDS = 120
MAX_RETRIES = 3
RETRY_TIMEOUT_SECONDS = 300

# 如使用系统代理，优先沿用urllib能读取的系统/环境代理。
# 如需手动指定，请填自己实际使用的HTTP代理地址，不要随意填写端口。
PROXY_URL = None
USE_SYSTEM_PROXY = True

# 可选：读取成功后导出原始距离GeoTIFF、300 dpi图、覆盖统计与环境记录。
EXPORT_RESULTS = True

APPLY_PARITY = True  # Original v1 condition; sensitivity cell below toggles explicitly.


## 4 HTTP连通性检查

In [ ]:
import json
import time
import urllib.request
from urllib.parse import urlsplit, quote

parsed = urlsplit(STORE_URL)
if parsed.scheme != "s3":
    raise ValueError("STORE_URL应为s3://bucket/path形式。")
HTTP_ARCHIVE_URL = STORE_ENDPOINT.rstrip("/") + "/" + parsed.netloc + parsed.path.rstrip("/")
metadata_url = HTTP_ARCHIVE_URL + "/zarr.json"
print("阶段1/4：HTTP元数据连通性检查", flush=True)
started = time.perf_counter()
proxy_handler = urllib.request.ProxyHandler({"http": PROXY_URL, "https": PROXY_URL}) if PROXY_URL else urllib.request.ProxyHandler(None if USE_SYSTEM_PROXY else {})
opener = urllib.request.build_opener(proxy_handler)
try:
    request = urllib.request.Request(metadata_url, headers={"User-Agent": "tourism-embedding-reproduction"})
    with opener.open(request, timeout=CONNECT_TIMEOUT_SECONDS) as response:
        metadata = json.load(response)
        print("HTTP状态：", response.status, "Zarr格式：", metadata.get("zarr_format"), flush=True)
    print(f"检查耗时：{time.perf_counter()-started:.1f}秒", flush=True)
except Exception as exc:
    raise RuntimeError("HTTP元数据读取失败。请先检查当前网络或代理；不必修改AOI和评分公式。") from exc


## 5 档案与GIS函数

In [ ]:
from dataclasses import dataclass
from typing import Any
import matplotlib.pyplot as plt
import numpy as np
import zarr
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.ticker import FuncFormatter
from obstore.store import S3Store
from pyproj import CRS, Transformer
from rasterio.features import geometry_mask
from rasterio.transform import Affine
from rasterio.windows import from_bounds
from shapely.geometry import Polygon, box, mapping
from shapely import transform as transform_geometry

@dataclass
class AoiEmbeddings:
    zone: int
    embeddings: np.ndarray
    inside: np.ndarray
    bounds: tuple[int, int, int, int]


@dataclass
class ZoneComparisonResult:
    aoi: AoiEmbeddings
    scores: np.ndarray


EMBEDDINGS_ARRAY = "embeddings"
NODATA = -128
CHUNK_SIZE = 256
GRID_WIDTH = 98304
GRID_HEIGHT = 1867776
GRID_TRANSFORM = Affine(10, 0, 0, 0, -10, 9502720)
CHANGE_SCALE_MAX = 0.5


def open_archive(store_url: str, endpoint: str) -> Any:
    from datetime import timedelta
    zarr.config.set({
        "async.concurrency": ASYNC_CONCURRENCY,
        "array.sharding_coalesce_max_gap_bytes": 0,
    })
    client_options = {
        "connect_timeout": timedelta(seconds=CONNECT_TIMEOUT_SECONDS),
        "timeout": timedelta(seconds=REQUEST_TIMEOUT_SECONDS),
    }
    proxy = PROXY_URL
    hostname = urlsplit(endpoint).hostname
    if not proxy and USE_SYSTEM_PROXY and hostname and not urllib.request.proxy_bypass(hostname):
        proxies = urllib.request.getproxies()
        proxy = proxies.get("https") or proxies.get("http")
    if proxy:
        client_options["proxy_url"] = proxy
    print("obstore HTTP代理：", "启用" if proxy else "未配置", flush=True)
    store = S3Store.from_url(
        store_url, endpoint=endpoint, skip_signature=True,
        client_options=client_options,
        retry_config={
            "max_retries": MAX_RETRIES,
            "retry_timeout": timedelta(seconds=RETRY_TIMEOUT_SECONDS),
        },
    )
    return zarr.open_group(
        zarr.storage.ObjectStore(store, read_only=True),
        mode="r", zarr_format=3, use_consolidated=False,
    )



LONGITUDE_ORIGIN = -180
UTM_ZONE_WIDTH = 6


def utm_zone(lon: float) -> int:
    return min(60, int((lon - LONGITUDE_ORIGIN) % 360 // UTM_ZONE_WIDTH) + 1)


def zone_group(zone: int) -> str:
    return f"utm{zone:02d}"


def zones_for_aoi(rings: list[list[list[float]]]) -> list[int]:
    longitudes = [lon for ring in rings for lon, _ in ring]
    if not longitudes:
        return []
    first = utm_zone(min(longitudes))
    last = utm_zone(max(longitudes))
    if first <= last:
        return list(range(first, last + 1))
    return list(range(first, 61)) + list(range(1, last + 1))


def zone_band(zone: int, aoi_west: float, aoi_east: float) -> tuple[float, float]:
    west = LONGITUDE_ORIGIN + (zone - 1) * UTM_ZONE_WIDTH
    middle = (aoi_west + aoi_east) / 2
    west += round((middle - west - UTM_ZONE_WIDTH / 2) / 360) * 360
    return west, west + UTM_ZONE_WIDTH


SOUTHERN_LATITUDE_LIMIT = -90
NORTHERN_LATITUDE_LIMIT = 90


def zone_longitude_bounds(
    zone: int, aoi_west: float, aoi_east: float, zones: list[int]
) -> tuple[float, float]:
    west, east = zone_band(zone, aoi_west, aoi_east)
    if zone == zones[0]:
        west = min(west, aoi_west)
    if zone == zones[-1]:
        east = max(east, aoi_east)
    return west, east


def aoi_geometry(
    rings: list[list[list[float]]], zone: int, zones: list[int]
) -> tuple[Any, Transformer]:
    crs = CRS.from_proj4(f"+proj=utm +zone={zone} +datum=WGS84 +units=m +no_defs")
    to_utm = Transformer.from_crs("EPSG:4326", crs, always_xy=True)
    polygon = Polygon(rings[0])
    for ring in rings[1:]:
        polygon = polygon.symmetric_difference(Polygon(ring))
    west, east = zone_longitude_bounds(zone, polygon.bounds[0], polygon.bounds[2], zones)
    clipped = polygon.intersection(
        box(west, SOUTHERN_LATITUDE_LIMIT, east, NORTHERN_LATITUDE_LIMIT)
    )
    if clipped.is_empty or clipped.area == 0:
        return None, to_utm
    return transform_geometry(clipped, to_utm.transform, interleaved=False), to_utm


def chunk_bounds(window: Any) -> tuple[int, int, int, int]:
    x0 = max(0, int(np.floor(window.col_off / CHUNK_SIZE) * CHUNK_SIZE))
    y0 = max(0, int(np.floor(window.row_off / CHUNK_SIZE) * CHUNK_SIZE))
    x1 = min(GRID_WIDTH, int(np.ceil((window.col_off + window.width) / CHUNK_SIZE) * CHUNK_SIZE))
    y1 = min(GRID_HEIGHT, int(np.ceil((window.row_off + window.height) / CHUNK_SIZE) * CHUNK_SIZE))
    return x0, y0, x1, y1


def year_index_for(available: np.ndarray, year: int) -> int:
    matches = np.flatnonzero(available == year)
    if not len(matches):
        raise ValueError(f"Year {year} is unavailable. Available years: {available.tolist()}")
    return int(matches[0])


def load_embeddings_for_aoi(
    archive: Any, rings: list[list[list[float]]], years: list[int], dims: int
) -> list[AoiEmbeddings]:
    results = []
    zones = zones_for_aoi(rings)
    for zone in zones:
        geometry, _ = aoi_geometry(rings, zone, zones)
        if geometry is None:
            continue
        window = from_bounds(*geometry.bounds, transform=GRID_TRANSFORM)
        x0, y0, x1, y1 = chunk_bounds(window)
        if x1 <= x0 or y1 <= y0:
            continue
        group = archive[zone_group(zone)]
        available = np.asarray(group["time"][:]).astype(int)
        array = group[EMBEDDINGS_ARRAY]
        print(f"UTM {zone}：窗口 {x1-x0}×{y1-y0}，{dims}维，年份 {years}", flush=True)
        loaded_years = []
        for year in years:
            started = time.perf_counter()
            print(f"  正在下载 {year} 年数据，网络重试期间可能暂时没有新输出……", flush=True)
            values = np.asarray(
                array[year_index_for(available, year), :dims, y0:y1, x0:x1],
                dtype=np.int8,
            )
            loaded_years.append(values)
            print(f"  {year} 年完成，耗时 {time.perf_counter()-started:.1f} 秒", flush=True)
        embeddings = np.stack(loaded_years)
        transform = GRID_TRANSFORM * Affine.translation(x0, y0)
        inside = geometry_mask(
            [mapping(geometry)], out_shape=(y1 - y0, x1 - x0), transform=transform, invert=True
        )
        results.append(AoiEmbeddings(zone, embeddings, inside, (x0, y0, x1, y1)))
    if not results:
        raise ValueError("The AOI does not overlap any UTM zone.")
    return results


## 6 分析函数（沿用原始导出公式）

In [ ]:
def unit_vectors(vectors: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(vectors, axis=0)
    return vectors / np.maximum(norms, 1e-12)[None, :, :]


def dequantize(values: np.ndarray) -> np.ndarray:
    scaled = values.astype(np.float32) / 127.5
    return np.sign(scaled) * np.abs(scaled) ** 2


PARITY_SAMPLE_CELLS = 16384
PARITY_MIN_PIXELS = 64


# The archive's encoder leaves a fixed offset per (row % 2, column % 2) grid position,
# which draws one-pixel stripes. Its size varies by region, so it is estimated per area.
def parity_offsets(embeddings: np.ndarray, x0: int, y0: int) -> np.ndarray:
    bands, height, width = embeddings.shape
    cells_down, cells_across = height // 2, width // 2
    step = max(1, int(np.ceil(np.sqrt(cells_down * cells_across / PARITY_SAMPLE_CELLS))))
    means = np.zeros((2, 2, bands), dtype=np.float32)
    for dy in (0, 1):
        for dx in (0, 1):
            sample = embeddings[:, dy:cells_down * 2:step * 2, dx:cells_across * 2:step * 2]
            sample = sample.reshape(bands, -1)
            sample = sample[:, sample[0] != NODATA]
            if sample.shape[1] < PARITY_MIN_PIXELS:
                return np.zeros((2, 2, bands), dtype=np.float32)
            means[(y0 + dy) % 2, (x0 + dx) % 2] = dequantize(sample).mean(axis=1)
    return means - means.mean(axis=(0, 1))


def corrected_vectors(embeddings: np.ndarray, x0: int, y0: int) -> np.ndarray:
    """Dequantize and optionally remove the notebook's regional parity offsets.

    APPLY_PARITY=False is the ablation condition; True reproduces the original
    exported workflow. No other processing step changes between conditions.
    """
    vectors = dequantize(embeddings)
    if not globals().get("APPLY_PARITY", True):
        return vectors
    offsets = parity_offsets(embeddings, x0, y0)
    for dy in (0, 1):
        for dx in (0, 1):
            vectors[:, dy::2, dx::2] -= offsets[(y0 + dy) % 2, (x0 + dx) % 2][:, None, None]
    return vectors

def score_years(embeddings: np.ndarray, inside: np.ndarray, x0: int, y0: int) -> np.ndarray:
    if VIEW == "pair" and len(YEARS) != 2:
        raise ValueError("Pair comparison requires exactly two years.")
    if VIEW not in ("pair", "churn", "when"):
        raise ValueError(f"Unknown comparison view: {VIEW}")
    scores = np.full(inside.shape, np.nan, dtype=np.float32)
    previous = np.zeros(embeddings.shape[1:], dtype=np.float32)
    previous_valid = np.zeros(inside.shape, dtype=bool)
    count = np.zeros(inside.shape, dtype=np.uint8)
    best_jump = np.full(inside.shape, -np.inf, dtype=np.float32)

    for index, values in enumerate(embeddings):
        valid = inside & ~np.all(values == NODATA, axis=0)
        current = unit_vectors(corrected_vectors(values, x0, y0))
        both = valid & previous_valid
        if np.any(both):
            jump = 1 - np.sum(previous * current, axis=0)
            if VIEW == "pair":
                scores[both] = jump[both]
            elif VIEW == "churn":
                scores[both & (count == 1)] = 0
                scores[both] += jump[both]
            else:
                better = both & (jump > best_jump)
                scores[better] = index
                best_jump[better] = jump[better]
        previous[:, valid] = current[:, valid]
        previous_valid |= valid
        count[valid] += 1
    return scores


def compare_years(data: list[AoiEmbeddings]) -> list[ZoneComparisonResult]:
    results = [
        ZoneComparisonResult(part, score_years(part.embeddings, part.inside, *part.bounds[:2]))
        for part in data
    ]
    if not any(np.isfinite(result.scores).any() for result in results):
        raise ValueError("No pixel in the AOI has data for at least two selected years.")
    return results


## 7 绘图函数

In [ ]:
def crop_to_aoi(
    scores: np.ndarray, inside: np.ndarray, bounds: tuple[int, int, int, int]
) -> tuple[np.ndarray, tuple[float, float, float, float]]:
    x0, y0, x1, y1 = bounds
    rows, columns = np.where(inside)
    if len(rows):
        row_start, row_stop = int(rows.min()), int(rows.max()) + 1
        col_start, col_stop = int(columns.min()), int(columns.max()) + 1
        scores = scores[row_start:row_stop, col_start:col_stop]
        x0, y0 = x0 + col_start, y0 + row_start
        x1, y1 = x0 + col_stop - col_start, y0 + row_stop - row_start
    left, top = GRID_TRANSFORM * (x0, y0)
    right, bottom = GRID_TRANSFORM * (x1, y1)
    return scores, (left, right, bottom, top)


def plot_results(results: list[ZoneComparisonResult]) -> Any:
    fig, axes = plt.subplots(
        1, len(results), figsize=(6 * len(results), 6), squeeze=False, layout="constrained"
    )
    valid_scores = np.concatenate([
        result.scores[np.isfinite(result.scores)] for result in results
    ])
    if VIEW == "when":
        cmap = ListedColormap(plt.get_cmap("tab20")(np.linspace(0, 1, len(YEARS))))
        cmap.set_bad("#e5e7eb")
        norm = BoundaryNorm(np.arange(len(YEARS) + 1) - 0.5, len(YEARS))
    else:
        cmap = "YlOrRd"
        norm = None
    color_min = 0 if VIEW == "pair" else float(np.min(valid_scores))
    color_max = CHANGE_SCALE_MAX if VIEW == "pair" else float(np.max(valid_scores))
    if color_max <= color_min:
        color_max = color_min + 1e-6
    for axis, result in zip(axes[0], results):
        display, extent = crop_to_aoi(
            result.scores, result.aoi.inside, result.aoi.bounds
        )
        image = axis.imshow(
            display, cmap=cmap, norm=norm,
            vmin=None if VIEW == "when" else color_min,
            vmax=None if VIEW == "when" else color_max,
            extent=extent, interpolation="nearest", aspect="equal",
        )
        axis.set_title(f"UTM zone {result.aoi.zone}")
        axis.set_xlabel("UTM easting (km)")
        axis.set_ylabel("UTM northing (km)")
        axis.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value / 1000:g}"))
        axis.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value / 1000:g}"))
        axis.set_facecolor("#e5e7eb")
    if VIEW == "when":
        bar = fig.colorbar(image, ax=list(axes[0]), ticks=np.arange(1, len(YEARS)), label="Year of largest change")
        bar.ax.set_yticklabels([str(year) for year in YEARS[1:]])
    else:
        label = "Change (1 − cosine)" if VIEW == "pair" else "Total change (Σ 1 − cosine)"
        fig.colorbar(image, ax=list(axes[0]), label=label)
    title = f"{YEARS[0]} → {YEARS[-1]}" if VIEW == "pair" else f"{YEARS[0]}–{YEARS[-1]}"
    fig.suptitle(f"{VIEW.capitalize()} comparison · {title}")
    plt.show()
    return fig


## 8 打开目录并加载数据

In [ ]:
print("阶段2/4：正在通过obstore打开远程目录……", flush=True)
try:
    archive = open_archive(STORE_URL, STORE_ENDPOINT)
except Exception as exc:
    raise RuntimeError(
        "obstore打开目录失败。若阶段1已返回HTTP 200，请检查obstore与系统代理的连接差异。"
        "指定Zarr v3已减少格式探测请求，但不能保证网络不超时。"
    ) from exc
print("远程目录打开成功。阶段3/4：正在读取AOI嵌入……", flush=True)
try:
    data = load_embeddings_for_aoi(archive, AOI, YEARS, EMBEDDING_DIMS)
except Exception as exc:
    raise RuntimeError(
        "目录已打开，但AOI数据读取失败；请查看下方原始异常。"
        "若是TimedOut，说明远程分块下载仍有网络问题，不代表计算完成。"
    ) from exc
print("Embeddings loaded.", flush=True)


## 9 计算变化

In [ ]:
print("阶段4/4：正在计算变化……", flush=True)
results = compare_years(data)
print("Analysis complete.", flush=True)


## 10 显示结果

In [ ]:
figure = plot_results(results)


## 11 导出运行结果

只有以上步骤全部成功，才会在当前工作目录的`outputs/`下新建独立子目录。单波段TIFF是本次重跑的原始评分，不是原浏览器RGBA图。有效评分比例不等于分类精度；图像300 dpi也不增加卫星空间分辨率。

In [ ]:
if EXPORT_RESULTS:
    from pathlib import Path
    from datetime import datetime, timezone
    import platform
    import subprocess
    import rasterio

    output_dir = Path("outputs") / ("beijing_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ"))
    output_dir.mkdir(parents=True, exist_ok=False)
    figure.savefig(output_dir / "beijing_comparison.png", dpi=300, bbox_inches="tight")
    summaries = []
    for result in results:
        aoi = result.aoi
        x0, y0, x1, y1 = aoi.bounds
        score = result.scores.astype(np.float32)
        destination = output_dir / f"beijing_{VIEW}_utm{aoi.zone}_scores.tif"
        with rasterio.open(
            destination, "w", driver="GTiff", height=score.shape[0], width=score.shape[1],
            count=1, dtype="float32", crs=f"EPSG:{32600+aoi.zone}",
            transform=GRID_TRANSFORM * Affine.translation(x0, y0),
            nodata=float("nan"), compress="deflate",
        ) as target:
            target.write(score, 1)
        denominator = int(aoi.inside.sum())
        valid = int((np.isfinite(score) & aoi.inside).sum())
        summaries.append({
            "zone": aoi.zone, "inside_aoi_pixels": denominator,
            "valid_score_pixels": valid,
            "valid_score_fraction": valid / denominator if denominator else None,
            "raster": destination.name,
        })
    record = {
        "case": "beijing", "aoi": AOI, "years": YEARS, "view": VIEW,
        "embedding_dims": EMBEDDING_DIMS, "store_url": STORE_URL,
        "store_endpoint": STORE_ENDPOINT,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "python": sys.version, "platform": platform.platform(),
        "versions": {m: version(m) for m in ("numpy", "zarr", "obstore", "matplotlib", "pyproj", "rasterio", "shapely")},
        "read_settings": {"concurrency": ASYNC_CONCURRENCY, "coalescing_gap_bytes": 0,
                          "request_timeout_seconds": REQUEST_TIMEOUT_SECONDS, "max_retries": MAX_RETRIES},
        "coverage": summaries,
        "limitations": "Numerical rerun outputs, not an independent accuracy or browser-agreement validation. The 300 dpi plot is not the browser basemap screenshot.",
    }
    (output_dir / "run_metadata.json").write_text(json.dumps(record, indent=2, ensure_ascii=False), encoding="utf-8")
    freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True)
    if freeze.returncode == 0:
        (output_dir / "environment.txt").write_text(freeze.stdout, encoding="utf-8")
    print("导出完成：", output_dir.resolve())
else:
    print("EXPORT_RESULTS=False，本次仅显示结果。")


## 12. JHTT sensitivity experiment: parity correction × embedding dimension

This extension preserves the original v1 cells and adds a controlled experiment. It reuses the loaded 64-D raw vectors, downloads 128-D vectors independently, and calculates four conditions: 64-D/128-D × parity off/on. It exports single-band numerical GeoTIFFs, compressed arrays, score summaries, parity-offset diagnostics, and rank-stability metrics. It does not produce accuracy statistics or ground-truth claims. The 128-D download requires the same live archive access as the original notebook.

In [ ]:
# EXPERIMENT MODULE: parity ablation and independent 64/128-D comparison
# Run after all preceding cells have completed successfully and the standard
# 64-D archive data are loaded in `data`. This cell performs fresh 128-D reads.
from pathlib import Path
from datetime import datetime, timezone
import json, csv, math
import numpy as np
import rasterio
from rasterio.transform import Affine
from scipy.stats import spearmanr

EXPERIMENT_OUT = Path("outputs") / ("parity_dim_sensitivity_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))
EXPERIMENT_OUT.mkdir(parents=True, exist_ok=False)
APPLY_PARITY = True

# Existing `data` was loaded at EMBEDDING_DIMS=64 in the original notebook.
data64 = data
print("Loading independent 128-dimensional vectors from the same archive...")
data128 = load_embeddings_for_aoi(archive, AOI, YEARS, 128)


def _score_pair(part, use_parity):
    global APPLY_PARITY
    APPLY_PARITY = bool(use_parity)
    e0, e1 = part.embeddings[0], part.embeddings[1]
    valid0 = part.inside & ~np.all(e0 == NODATA, axis=0)
    valid1 = part.inside & ~np.all(e1 == NODATA, axis=0)
    valid = valid0 & valid1
    v0 = unit_vectors(corrected_vectors(e0, part.bounds[0], part.bounds[1]))
    v1 = unit_vectors(corrected_vectors(e1, part.bounds[0], part.bounds[1]))
    score = np.full(part.inside.shape, np.nan, dtype=np.float32)
    if valid.any():
        jump = 1.0 - np.sum(v0 * v1, axis=0)
        score[valid] = jump[valid]
    return score, valid


def _offset_diagnostics(part):
    rows=[]
    for yi, year in enumerate(YEARS):
        offsets=parity_offsets(part.embeddings[yi], part.bounds[0], part.bounds[1])
        norms=np.linalg.norm(offsets, axis=2)
        for r in range(2):
            for c in range(2):
                rows.append({"year":int(year),"row_parity":r,"col_parity":c,
                             "offset_l2_norm":float(norms[r,c]),
                             "offset_mean_abs":float(np.mean(np.abs(offsets[r,c]))),
                             "offset_max_abs":float(np.max(np.abs(offsets[r,c])))})
    return rows


def _rank_metrics(a,b):
    ok=np.isfinite(a)&np.isfinite(b)
    x=a[ok].astype(np.float64); y=b[ok].astype(np.float64)
    if len(x)<3:
        return {"n_common":int(len(x)),"spearman_rho":None,"spearman_p":None,
                "top1_jaccard":None,"top5_jaccard":None,"top10_jaccard":None}
    sp=spearmanr(x,y)
    out={"n_common":int(len(x)),"spearman_rho":float(sp.statistic),"spearman_p":float(sp.pvalue)}
    for pct in (1,5,10):
        k=max(1,int(math.ceil(len(x)*pct/100)))
        ia=set(np.argpartition(x,-k)[-k:].tolist())
        ib=set(np.argpartition(y,-k)[-k:].tolist())
        out[f"top{pct}_jaccard"]=len(ia&ib)/len(ia|ib) if ia|ib else None
    return out

all_scores={}
summary=[]
offsets=[]
for dims, parts in ((64,data64),(128,data128)):
    for use_parity in (False,True):
        label=f"d{dims}_"+("parity" if use_parity else "no_parity")
        print("Scoring",label)
        case_scores=[]
        for part in parts:
            score,valid=_score_pair(part,use_parity)
            case_scores.append((part,score,valid))
            vals=score[valid]
            summary.append({"case":"beijing","utm_zone":part.zone,"dimensions":dims,
                "parity_correction":use_parity,"valid_pixels":int(valid.sum()),
                "mean":float(np.mean(vals)),"median":float(np.median(vals)),
                "sd":float(np.std(vals)),"p95":float(np.quantile(vals,.95)),
                "min":float(np.min(vals)),"max":float(np.max(vals))})
            offsets.extend([{**r,"case":"beijing","dimensions":dims} for r in _offset_diagnostics(part)])
            # Export numerical score raster for independent inspection.
            x0,y0,x1,y1=part.bounds
            transform=GRID_TRANSFORM*Affine.translation(x0,y0)
            tif=EXPERIMENT_OUT/f"beijing_{label}_utm{part.zone}.tif"
            with rasterio.open(tif,"w",driver="GTiff",height=score.shape[0],width=score.shape[1],
                count=1,dtype="float32",crs=f"EPSG:{32600+part.zone}",transform=transform,
                nodata=float("nan"),compress="deflate") as dst:
                dst.write(score,1)
            np.savez_compressed(EXPERIMENT_OUT/f"beijing_{label}_utm{part.zone}.npz",
                                score=score,valid=valid)
        all_scores[label]=case_scores

# Pairwise ranking stability, computed on the exact intersection of valid pixels.
comparisons=[]
for dims in (64,128):
    a=all_scores[f"d{dims}_no_parity"]; b=all_scores[f"d{dims}_parity"]
    for (pa,sa,va),(pb,sb,vb) in zip(a,b):
        m=_rank_metrics(np.where(va&vb,sa,np.nan),np.where(va&vb,sb,np.nan))
        comparisons.append({"comparison":f"{dims}D: parity vs no-parity","case":"beijing","utm_zone":pa.zone,**m})
for parity in (False,True):
    a=all_scores[f"d64_{'parity' if parity else 'no_parity'}"]
    b=all_scores[f"d128_{'parity' if parity else 'no_parity'}"]
    for (pa,sa,va),(pb,sb,vb) in zip(a,b):
        m=_rank_metrics(np.where(va&vb,sa,np.nan),np.where(va&vb,sb,np.nan))
        comparisons.append({"comparison":f"64D vs 128D: {'with' if parity else 'without'} parity","case":"beijing","utm_zone":pa.zone,**m})

# Replace case placeholder from AOI location, rather than relying on notebook filename.
case_name="beijing" if AOI[0][0][0] < 117 else "hainan"
for row in summary+offsets+comparisons:
    if row.get("case")=="beijing": row["case"]=case_name
for filename,rows in (("score_summary.csv",summary),("parity_offset_diagnostics.csv",offsets),("rank_stability.csv",comparisons)):
    if rows:
        with open(EXPERIMENT_OUT/filename,"w",newline="",encoding="utf-8-sig") as f:
            w=csv.DictWriter(f,fieldnames=list(rows[0]));w.writeheader();w.writerows(rows)
metadata={"case":case_name,"years":YEARS,"dimensions_tested":[64,128],
 "parity_conditions":["official dequantization without regional parity correction","same dequantization with original notebook parity correction"],
 "archive":STORE_URL,"endpoint":STORE_ENDPOINT,"created_utc":datetime.now(timezone.utc).isoformat(),
 "important":"This is a numerical sensitivity analysis, not accuracy validation. 64D and 128D are analyzed independently. Compare only on common valid pixels."}
(EXPERIMENT_OUT/"experiment_metadata.json").write_text(json.dumps(metadata,indent=2),encoding="utf-8")
print("EXPERIMENT COMPLETE:",EXPERIMENT_OUT.resolve())
print("Review score_summary.csv, parity_offset_diagnostics.csv, and rank_stability.csv.")
